In [43]:
import requests
import json
import pandas as pd
import os
from tqdm import tqdm

D:\Anaconda\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\Anaconda\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [44]:
class whoAmI:
    def __init__(self):
        self.getDim='https://ghoapi.azureedge.net/api/Dimension'
        self.getDimValues='https://ghoapi.azureedge.net/api/DIMENSION/COUNTRY/DimensionValues'
        self.getIndValues='https://ghoapi.azureedge.net/api/Indicator'
        self.getIndFilterVal='https://ghoapi.azureedge.net/api/Indicator?$filter=contains(IndicatorName,'
        self.getIndFilterSpecVal='https://ghoapi.azureedge.net/api/Indicator?$filter=IndicatorName eq '
        self.getIndData='https://ghoapi.azureedge.net/api/'
urls = whoAmI()

In [ ]:
DATASET_SAVED = "../../data/raw_official"
URL_INDICATORS = "../../data/urls"

In [46]:
# Khởi tạo thư mục nếu chưa tồn tại
os.makedirs(DATASET_SAVED, exist_ok=True)
os.makedirs(URL_INDICATORS, exist_ok=True)

In [47]:
# Lấy các bộ dữ liệu đã có
existed = set()
for file in os.listdir(DATASET_SAVED):
    indicator = file.split('.')[0]
    existed.add(indicator)

In [48]:
dataset_indicators = set()
for file in os.listdir(URL_INDICATORS):
    with open(URL_INDICATORS + '/' + file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line in existed:
                continue
            dataset_indicators.add(line)
print("Các indicators:", dataset_indicators)

Các indicators: {'R_Price_mp_estimate_ppp', 'W24_exp_date_C', 'W14_harm_effects_B', 'TOBACCO_MPOWER_E1_DIRECTADBANS', 'R_afford_inflation_adjusted_price_ppp', 'P3_universities', 'W23_emissions_front_back_B', 'R_afford_tax_auto_adjust', 'TOBACCO_MPOWER_R_INFLATIONADJUSTEDPRICES', 'R_Other_average', 'E_compl_e12', 'E14_compliance', 'Camp_gov_prog', 'W6_font_C', 'W21_emissions_quant_C', 'W19_colours_numbers_B', 'E14_prod_tv_films', 'E9_free_distrib', 'P_count_places_sf', 'Adult_curr_tob_smoking', 'P10_subnat_auth_exists', 'W25_quitline_number_C', 'R_admin_duty_free_banned', 'W11_outside_pack_C', 'P_comprehensive_subnat', 'W20_flavours_B', 'Yth_daily_tob_use', 'TOB_R_PRICE', 'Yth_curr_tob_smoking', 'TOBACCO_MPOWER_W2C_TOBACCOSMOKELESS', 'Adult_daily_cig_smoking', 'Rev_type', 'W12_imports_dutyfree_B', 'R_admin_tax_stamps_cig', 'P5_compliance', 'R_min_price_policy', 'P2_compliance', 'W20_flavours_A', 'W21_emissions_quant_A', 'W3_pc_front_A', 'P_Group', 'Rev_import', 'W24_exp_date_A', 'R_Ad_v

In [ ]:
def request_data(indicator):
    resp = requests.get(urls.getIndData + '/' + indicator)
    data = json.loads(resp.content)["value"]
    fields = ['ParentLocationCode', 'SpatialDim', 'Dim1', 'Value', 'NumericValue', 'TimeDimensionBegin', 'TimeDimensionEnd', 'TimeDimension', 'IndicatorCode']

    records = []
    for row in data:
        _tempo = dict()
        for field in fields:
            _tempo[field] = row[field]
        records.append(_tempo)

    with open(DATASET_SAVED + '/' + indicator + '.json', "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [50]:
# Hàm lấy hết dữ liệu
def retrieve_all():
    for indicator in tqdm(dataset_indicators):
        try:
            request_data(indicator)
        except Exception as e:
            print("Error:", e)

In [51]:
retrieve_all()

100%|██████████| 259/259 [14:58<00:00,  3.47s/it]


In [ ]:
# Biến chỉ định lưu trữ
CONCAT_GROUP = "../../data/raw_concat"
os.makedirs(CONCAT_GROUP, exist_ok=True)

In [53]:
# Tiến hành tổ chức và nhóm lại dữ liệu vào file gốc theo các nhãn
def prepare_data_through_group(group):
    # Group là nhãn dữ liệu, tên thư mục
    target = group.split('.')[0]
    objectives = set()

    with open(URL_INDICATORS + '/' + group, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            objectives.add(line)
    
    array_data = []
    for file in tqdm(os.listdir(DATASET_SAVED)):
        _label = file.split('.')[0]
        if _label in objectives:
            # Đúng nhãn dữ liệu, tiến hành load ra và ghép lại thành một file hoàn chỉnh
            with open(DATASET_SAVED + '/' + file, "rb") as f:
                data = json.load(f)
                
                # Tiến hành nối dữ liệu vào file
            array_data = array_data + data
                
    concated_result = CONCAT_GROUP + '/' + target + '.json'
    
    print("Tổng số điểm dữ liệu là:", len(array_data))
    with open(concated_result, "w", encoding="utf-8") as f:
        json.dump(array_data, f, ensure_ascii=False, indent=2)
    print("Hoàn thành xử lý:", target)

In [54]:
# Chạy xử lí chỉ số BMI
prepare_data_through_group("BMI.txt")

100%|██████████| 546/546 [00:00<00:00, 863.93it/s]


Tổng số điểm dữ liệu là: 453077
Hoàn thành xử lý: BMI


In [55]:
# CHạy xử lí chỉ số Diabetes
prepare_data_through_group("diabetes.txt")

100%|██████████| 546/546 [00:00<00:00, 1895.16it/s]


Tổng số điểm dữ liệu là: 211564
Hoàn thành xử lý: diabetes


In [56]:
# Chạy xử lí tiêu thụ cồn
prepare_data_through_group("alcohol_consumption.txt")

100%|██████████| 546/546 [00:01<00:00, 499.63it/s] 


Tổng số điểm dữ liệu là: 297817
Hoàn thành xử lý: alcohol_consumption


In [57]:
# Chạy xử lí air_pollution
prepare_data_through_group("air_pollution.txt")

100%|██████████| 546/546 [00:00<00:00, 605.98it/s]


Tổng số điểm dữ liệu là: 491076
Hoàn thành xử lý: air_pollution


In [58]:
# Chạy xử lí cholesterol
prepare_data_through_group("cholesterol.txt")

100%|██████████| 546/546 [00:00<00:00, 3615.86it/s]

Tổng số điểm dữ liệu là: 141804


Hoàn thành xử lý: cholesterol


In [59]:
# Chạy xử lí Glucose
prepare_data_through_group("glucose.txt")

100%|██████████| 546/546 [00:00<00:00, 25996.28it/s]

Tổng số điểm dữ liệu là: 21630
Hoàn thành xử lý: glucose


In [60]:
# Chạy xử lí hoạt động thể chất
prepare_data_through_group("physical_activities.txt")

100%|██████████| 546/546 [00:00<00:00, 15158.63it/s]

Tổng số điểm dữ liệu là: 35523


Hoàn thành xử lý: physical_activities


In [61]:
# Chạy xử lí tiếp cận cơ sở y tế
prepare_data_through_group("infrastructure.txt")

100%|██████████| 546/546 [00:00<00:00, 2689.65it/s]


Tổng số điểm dữ liệu là: 170916
Hoàn thành xử lý: infrastructure


In [62]:
# Chạy xử lí thuốc lá
prepare_data_through_group("tobacco.txt")

100%|██████████| 546/546 [00:07<00:00, 76.68it/s] 


Tổng số điểm dữ liệu là: 707018
Hoàn thành xử lý: tobacco


In [63]:
# Chạy xử lí bệnh tim mạch
prepare_data_through_group("cardiovascular_diseases.txt")

100%|██████████| 546/546 [00:00<00:00, 1573.79it/s]


Tổng số điểm dữ liệu là: 236876
Hoàn thành xử lý: cardiovascular_diseases
